# Spectral Bias in Diffusion Models — Session Summary (Apr 14–15, 2026)

This notebook documents the full pipeline built and experiments run across the April 14–15 session.
It covers:
1. DINOv2-B covariance computation and eigenspectrum analysis on ImageNet-1K
2. Architecture and loss design for Step 1 validation
3. Per-sigma analytic computation
4. d=50 gradient optimization sanity check across β schedules
5. DINOv2 Gaussian pilot training (in progress)

**Cluster:** Kempner Institute (FASRC), A100/H100 nodes, `torch2` conda env, `$STORE_DIR=/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang`

---
## 0. Imports and Paths

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display

STORE = "/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang"
REPO  = "/n/home12/binxuwang/Github/Loss-Weighting-Spectral-Bias"
COV_DIR  = f"{STORE}/DL_Projects/DINOv2_ImageNet1k_Covariance"
FIG_DIR  = f"{REPO}/figures/dinov2_covariance"
STEP1_SIMPLE = f"{STORE}/step1_results/simple_test_d50_alpha1"
STEP1_PILOT  = f"{STORE}/step1_results/mlp_dinov2"

sys.path.insert(0, REPO)
sys.path.insert(0, "/n/home12/binxuwang/Github/DiffusionLearningCurve")

print("Repo:", REPO)
print("Store:", STORE)

---
## 1. DINOv2-B Covariance on ImageNet-1K

We computed mean/covariance of DINOv2-B (with registers) features across 7 layers × 4 token types.

**Setup:**
- Model: `facebook/dinov2-vitb14-reg` (768-dim, registers)
- Token layout: CLS(0), reg(1–4), patch(5–260)
- Layers extracted: block1, block3, block5, block7, block9, block11, norm
- Data: ILSVRC 1.28M images at fp16, batch 256, IO ~1176 img/s
- Covariance estimated online in fp32 (normalized running mean/E[xxT] to avoid overflow)
- Runtime: ~26 minutes on A100

**Files saved:** `$STORE/DL_Projects/DINOv2_ImageNet1k_Covariance/{block1,block3,...,norm}.pt`
Each file: `{cls, reg, patch, all} × {mean, cov, n}`, size ~9 MB

In [ ]:
# Load norm layer patch covariance and compute eigenspectrum
stats = torch.load(f"{COV_DIR}/norm.pt", weights_only=True)
cov_patch = stats["patch"]["cov"].float()  # [768, 768]
mean_patch = stats["patch"]["mean"].float()  # [768]

eigval, eigvec = torch.linalg.eigh(cov_patch)
eigval = eigval.flip(0).numpy()  # descending
eigvec = eigvec.flip(1)          # [768, 768] columns = eigenvectors

print(f"norm/patch: mean norm = {mean_patch.norm():.3f}")
print(f"λ_1={eigval[0]:.4f}, λ_10={eigval[9]:.4f}, λ_100={eigval[99]:.4f}, λ_768={eigval[767]:.6f}")
print(f"Top-100 modes explain {100*eigval[:100].sum()/eigval.sum():.1f}% of variance")

In [ ]:
# Power-law fit on top 200 modes
from scipy import stats as spstats

k = np.arange(1, len(eigval)+1)
slope, intercept, r, _, _ = spstats.linregress(np.log(k[:200]), np.log(eigval[:200]))
alpha_data = -slope
print(f"Power-law fit (top 200 modes): α_data = {alpha_data:.3f}, R² = {r**2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(k, eigval, 'b.', ms=2, alpha=0.5)
axes[0].loglog(k[:200], np.exp(intercept)*k[:200]**slope, 'r-', lw=2,
                label=f'α={alpha_data:.2f}')
axes[0].set(xlabel='Mode k', ylabel='Eigenvalue λ_k',
            title='DINOv2 norm/patch eigenspectrum')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(k, 100*np.cumsum(eigval)/eigval.sum(), 'b-')
axes[1].axhline(90, color='r', ls='--', alpha=0.5, label='90%')
axes[1].set(xlabel='Modes', ylabel='Cumulative variance (%)',
            title='Cumulative variance explained')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Show pre-computed analysis figures
for fname, title in [
    ('spec_loglog.png',     'Log-log eigenspectrum across layers'),
    ('tok_compare_loglog.png', 'Token type comparison (CLS/reg/patch)'),
    ('tok_compare_slope.png',  'Power-law slope α across layers and tokens'),
]:
    path = f"{FIG_DIR}/{fname}"
    if os.path.exists(path):
        print(f"\n{title}")
        display(Image(path, width=700))

**Key findings:**
- norm/patch has power-law slope **α ≈ 0.56** (gentle, broad-spectrum)
- CLS token is most low-rank (steeper slope ~0.8–1.0)
- Register tokens are ultra-low-rank in early layers (1–3 dims explain 90% variance)
- We use **norm/patch** as our reference feature distribution

---
## 2. Architecture: MLPDenoiser

We replaced the original `LinearDenoiserShared` (sigma-blind, single W matrix) with `MLPDenoiser`, a sigma-conditioned MLP based on `UNetBlockStyleMLP_backbone` from DiffusionLearningCurve.

**Why LinearDenoiserShared was wrong:** It uses a single W matrix shared across all σ. The denoiser should learn different mappings at different noise levels — a sigma-blind denoiser cannot represent the correct score function.

**MLPDenoiser architecture:**
```
x_in  = x_noisy / sqrt(1 + σ²)           # input scaling
t_enc = log(σ) / 4                         # noise embedding
out   = UNetBlockStyleMLP(x_in, t_enc)
      = Linear(d→nhidden)
      + nlayers × UNetMLPBlock(nhidden, nhidden, time_embed_dim)
          [LN → fc → SiLU + adaptive scale+shift from σ-embed + residual]
      + Linear(nhidden→d)
```
σ is embedded via `GaussianFourierProjection` inside the backbone.

In [ ]:
from step1_validation.models import MLPDenoiser
from step1_validation.config import SIGMA_0, SIGMA_T

# d=768 pilot model
model_pilot = MLPDenoiser(ndim=768, nhidden=512, nlayers=6, time_embed_dim=64,
                           sigma_min=SIGMA_0, sigma_max=SIGMA_T)
n_params = sum(p.numel() for p in model_pilot.parameters())
print(f"Pilot model (d=768, nhidden=512, nlayers=6): {n_params:,} parameters")

# d=50 sanity check model
model_small = MLPDenoiser(ndim=50, nhidden=128, nlayers=3, time_embed_dim=32,
                           sigma_min=SIGMA_0, sigma_max=SIGMA_T)
n_params_small = sum(p.numel() for p in model_small.parameters())
print(f"Small model (d=50, nhidden=128, nlayers=3): {n_params_small:,} parameters")

# Test forward pass
x = torch.randn(4, 768)
sigma = torch.full((4,), 1.0)
with torch.no_grad():
    out = model_pilot(x, sigma)
print(f"Forward pass: x {x.shape} → out {out.shape}")

---
## 3. Loss Function: DeterministicPowerLawLoss

Instead of sampling σ randomly at each step, we use a **deterministic K-sigma grid** (K=50–100 log-spaced values) evaluated at every training step.

**Loss:**
$$\mathcal{L}(\theta) = \frac{1}{K} \sum_{j=1}^{K} w_{\text{norm}}(\sigma_j) \cdot \mathbb{E}_{x, \epsilon} \|D_\theta(x + \sigma_j \epsilon, \sigma_j) - x\|^2$$

where $w(\sigma) = \sigma^\beta$ and $w_{\text{norm}} = w / A_{\max}$ with $A_{\max} = \text{mean}(w(\sigma_j)(\lambda_{\max} + \sigma_j^2))$.

**Vectorization:** noisy inputs are stacked as `[K×N, d]` and passed in one forward call:
```python
x_flat     = X_noisy.reshape(K * N, d)          # [K*N, d]
sigma_flat = sigmas.view(K,1).expand(K,N).reshape(K*N)  # [K*N]
D_out      = net(x_flat, sigma_flat).reshape(K, N, d)
```

In [ ]:
from step1_validation.losses import DeterministicPowerLawLoss

betas = [-2.0, -1.0, 0.0, 1.0, 2.0]
sigma_grid = torch.logspace(np.log10(SIGMA_0), np.log10(SIGMA_T), 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(betas)))

for beta, c in zip(betas, colors):
    loss_fn = DeterministicPowerLawLoss(beta=beta, K_sigma=100, lambda_max=14.95)
    w = loss_fn.weights.numpy()
    axes[0].plot(sigma_grid.numpy(), w, color=c, label=f'β={beta:+.1f}', lw=2)
    axes[1].plot(sigma_grid.numpy(), w, color=c, label=f'β={beta:+.1f}', lw=2)

axes[0].set(xlabel='σ', ylabel='w_norm(σ)', title='Loss weights (linear)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[1].set(xlabel='σ', ylabel='w_norm(σ)', title='Loss weights (log-log)',
            xscale='log', yscale='log')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
plt.suptitle('DeterministicPowerLawLoss: normalized weights w_norm(σ) = σ^β / A_max')
plt.tight_layout()
plt.show()

---
## 4. Sampling-Based Evaluation

**Why not Jacobians?** The Jacobian gives a linear approximation of the denoiser, which is exact for the linear (Gaussian) case but doesn't test the actual generative quality. We use **sample covariance** from the ODE sampler instead.

**Evaluation pipeline:**
1. Generate `n_eval` samples via `edm_sampler` (Heun 2nd-order ODE, 20–50 steps)
2. Compute sample covariance: `cov = torch.cov(x_gen.T)`
3. Project to data eigenbasis: `var_k = (U.T @ cov @ U).diag()`
4. Track ratio `var_k(τ) / λ_k` over training steps τ

**Log-scale callback:** samples are saved at geometrically-spaced training steps (dense early, sparse late), matching the DiffusionLearningCurve pattern.

**Important note on `edm_sampler`:** the function signature is
```python
x_next = latents * t_steps[0]   # latents ~ N(0,I), scaled by sigma_max internally
```
So pass **unscaled** `torch.randn(n, d)` — the sampler scales by σ_max itself.

---
## 5. Per-Sigma Analytic Computation

Theory predicts emergence steps via the Φ integral:
$$\tau_k^* \approx \left(\frac{1}{\eta \lambda_k}\right)^{\alpha_\Phi(\beta)}$$

where $\alpha_\Phi$ is computed by numerically integrating $\Psi_k(\sigma)$ over the sigma grid.

**Results for d=200, α_data ∈ {0.56, 1.0, 2.0}, β ∈ [-2.5, 2.0]:**

In [ ]:
import json

with open(f"{REPO}/step1_validation/per_sigma_analytic_d200_v2.json") as f:
    results_v2 = json.load(f)

print(f"Loaded {len(results_v2)} configurations")
print("\nβ       | α_data=0.56 | α_data=1.0 | α_data=2.0")
print("--------|-------------|------------|------------")

# Group by beta
from collections import defaultdict
by_beta = defaultdict(dict)
for r in results_v2:
    by_beta[r['beta']][r['alpha_data']] = r.get('alpha_phi', float('nan'))

for beta in sorted(by_beta.keys()):
    a056 = by_beta[beta].get(0.56, float('nan'))
    a10  = by_beta[beta].get(1.0,  float('nan'))
    a20  = by_beta[beta].get(2.0,  float('nan'))
    print(f"  {beta:+.1f}  |  {a056:8.4f}   |  {a10:7.4f}  |  {a20:7.4f}")

**Key observation:** α_Φ saturates near 1.0 for β ≥ 0.5, while the heuristic formula α_Φ = 1 + β/2 predicts values > 1. The Φ integral is the correct prediction; the heuristic only holds in specific regimes.

**β=2.5 diagnosis:** w(σ_max=80)^2.5 ≈ 5.7×10⁴, dynamic range 3.2×10¹¹. Only 1/200 modes emerges within τ_max=10⁶ → removed from sweep. See `DIAGNOSIS_beta2p5_nan.md`.

---
## 6. DINOv2 Gaussian Dataset

Instead of real images, we train on **synthetic Gaussian samples** drawn from the DINOv2 feature distribution:
- Mean and covariance from norm/patch (d=768)
- N_train = 10,000 samples
- This gives a ground-truth Gaussian target where theory is exact

In [ ]:
from step1_validation.dinov2_gaussian_dataset import make_dinov2_gaussian_dataset

dataset = make_dinov2_gaussian_dataset(n_samples=10000, layer="norm", token="patch", seed=42)
X_train = dataset["X_train"]
eigval_dinov2 = dataset["eigval"].numpy()
eigvec_dinov2 = dataset["eigvec"]

print(f"X_train shape: {X_train.shape}")
print(f"λ_max={eigval_dinov2[0]:.4f}, λ_min={eigval_dinov2[-1]:.6f}")
print(f"Effective dimensionality: {(eigval_dinov2.sum()**2 / (eigval_dinov2**2).sum()):.1f}")

# Verify sample covariance matches target
cov_sample = torch.cov(X_train.T)
var_k_sample = (eigvec_dinov2.float().T @ cov_sample @ eigvec_dinov2.float()).diag().numpy()
ratio = var_k_sample / (eigval_dinov2 + 1e-10)
print(f"Sample covariance check: var_k/λ_k mean={ratio[:100].mean():.3f} (should be ~1.0)")

---
## 7. Sanity Check: d=50 Power-Law Gaussian

Before the full DINOv2 experiment, we ran a d=50 synthetic Gaussian experiment to verify:
1. Gradient optimization is stable for all β values
2. The covariance evaluation pipeline works correctly
3. Different β schedules show different convergence patterns

**Setup:** d=50, α_data=1.0 (eigenvalues λ_k = k⁻¹), MLPDenoiser(nhidden=128, nlayers=3), 5000 steps, A100 H100, ~3 minutes.

In [ ]:
# Load saved results
d, alpha = 50, 1.0
k_arr = np.arange(1, d+1, dtype=float)
eigval_50 = k_arr**(-alpha)

betas = [-2.0, -1.0, 0.0, 1.0, 2.0]
tags  = ["betam20","betam10","betap00","betap10","betap20"]

results = {}
for beta, tag in zip(betas, tags):
    path_steps = f"{STEP1_SIMPLE}/steps_{tag}.npy"
    if os.path.exists(path_steps):
        results[beta] = {
            "steps":    np.load(f"{STEP1_SIMPLE}/steps_{tag}.npy"),
            "loss":     np.load(f"{STEP1_SIMPLE}/loss_{tag}.npy"),
            "var_traj": np.load(f"{STEP1_SIMPLE}/var_traj_{tag}.npy"),
        }
    else:
        print(f"  Missing: {tag}")

print(f"Loaded {len(results)} beta configs: {list(results.keys())}")

# Summary table
print("\nβ     | final_loss | mean var_k/λ_k | modes(>0.5)/50 | modes(0.8-1.2)/50")
print("------|------------|----------------|----------------|------------------")
for beta, res in results.items():
    ratio_final = res["var_traj"][-1] / (eigval_50 + 1e-10)
    n_over05 = int((ratio_final > 0.5).sum())
    n_correct = int(((ratio_final > 0.8) & (ratio_final < 1.2)).sum())
    print(f"  {beta:+.1f} | {res['loss'][-1]:10.4f} | {ratio_final.mean():14.3f} | {n_over05:14d} | {n_correct:18d}")

In [ ]:
# Loss curves
colors_b = {-2.0:"#d62728",-1.0:"#ff7f0e",0.0:"#2ca02c",1.0:"#1f77b4",2.0:"#9467bd"}
fig, ax = plt.subplots(figsize=(9, 4))
for beta, res in results.items():
    ax.semilogy(res["steps"], res["loss"], color=colors_b[beta], label=f'β={beta:+.1f}', lw=2)
ax.set(xlabel='Training step', ylabel='Loss (log)', title='Loss curves — d=50, α=1.0')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Covariance convergence — log-log axes (the correct view)
mode_idx = np.unique(np.geomspace(1, d-1, 8).astype(int))
cmap = plt.cm.plasma

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (beta, res) in zip(axes, results.items()):
    steps, var_traj = res["steps"], res["var_traj"]
    for i, k_i in enumerate(mode_idx):
        ratio_k = var_traj[:, k_i] / (eigval_50[k_i] + 1e-10)
        ax.semilogy(steps+1, ratio_k, color=cmap(i/len(mode_idx)),
                    lw=1.5, alpha=0.85, label=f'k={k_i} λ={eigval_50[k_i]:.3f}')
    ax.axhline(1.0, color='gray', ls='--', lw=1.5, label='target')
    ax.axhline(0.5, color='red',  ls=':',  lw=1)
    ax.set(xlabel='Step (log)', ylabel='var_k/λ_k', title=f'β={beta:+.1f}',
           xscale='log'); ax.grid(True, alpha=0.3, which='both')
    if ax is axes[0]: ax.legend(fontsize=7, ncol=2)
axes[-1].axis('off')
plt.suptitle('Eigenmode convergence — d=50, α=1.0 | log-log axes | target=dashed at 1.0')
plt.tight_layout(); plt.show()

### Key Findings from d=50 Sanity Check

| β | Behaviour | Root cause |
|---|-----------|------------|
| **-2.0** | var_k/λ_k → **67×** (diverges) | Learns only small-σ denoising (trivial D≈x). Large-σ ODE steps generate ≈N(0,4I) — isotropic noise carries over. |
| **-1.0** | ~1.4× (slight overestimate) | Partial large-σ learning; some residual large-scale noise |
| **0.0** | **Sequential convergence to 1.0** ✓ | Uniform weighting teaches all σ levels. Mode k=0 emerges first, then k=1, k=2, ... as predicted by theory. |
| **+1.0** | Only k=0 converges (~step 2000) | Focuses on large σ; fine structure (small σ) is learned slowly |
| **+2.0** | Nothing converges in 5000 steps | Almost all signal at large σ; ODE refinement steps terrible |

**Critical insight:** Loss value ≠ covariance quality. β=+2.0 achieves the **lowest loss** (0.0011) but generates samples with near-zero variance. β=-2.0 achieves moderate loss (0.0035) but generates samples with 67× too much variance.

**For theory comparison:** Only β=0.0 converges correctly. The sequential emergence in the β=0.0 log-log plot shows the power-law relationship between emergence time and eigenvalue index — the core signal the paper's theory should predict.

In [ ]:
# Show the heatmap (pre-computed with LogNorm)
for fname, title in [
    ('cov_heatmap_log.png', 'Covariance heatmap — log colorscale (green=1.0, deep-green>1, red<1)'),
    ('cov_lines_loglog.png','Convergence lines — log-log axes'),
]:
    path = f"{STEP1_SIMPLE}/{fname}"
    if os.path.exists(path):
        print(f'\n{title}')
        display(Image(path, width=800))

---
## 8. DINOv2 Gaussian Pilot Training (d=768, in progress)

**Experiment:** Train MLPDenoiser(nhidden=512, nlayers=6) on DINOv2 norm/patch Gaussian (d=768, N=10k) for β ∈ {-2.0, -1.0, 0.0, +1.0}, 50k steps.

**Log-scale callback:** samples saved at 200 geometrically-spaced steps (dense early, sparse late), mirroring DiffusionLearningCurve.

**Status:** Running as Slurm array job 5769619 on `kempner_h100` partition.

In [ ]:
import glob

beta_dirs = {
    "-2.0": f"{STEP1_PILOT}/norm_patch_betam20_nhid512_nl6_seed42",
    "-1.0": f"{STEP1_PILOT}/norm_patch_betam10_nhid512_nl6_seed42",
     "0.0": f"{STEP1_PILOT}/norm_patch_betap00_nhid512_nl6_seed42",
    "+1.0": f"{STEP1_PILOT}/norm_patch_betap10_nhid512_nl6_seed42",
}

print("Pilot run progress:")
print(f"{'β':6s} | {'sample files':13s} | {'latest step':12s}")
print("-"*40)
for beta_str, bdir in beta_dirs.items():
    samples = sorted(glob.glob(f"{bdir}/samples/samples_step_*.pt"))
    if samples:
        latest = int(os.path.basename(samples[-1]).replace("samples_step_","").replace(".pt",""))
        print(f"  {beta_str:4s} | {len(samples):13d} | step {latest:>8d}")
    else:
        print(f"  {beta_str:4s} | {'no data':>13s} |")

In [ ]:
# Plot covariance trajectory for available pilot data
def load_var_traj(bdir, eigvec_t, n_modes=200):
    """Load sample files, compute var_k in eigenbasis."""
    samples_files = sorted(glob.glob(f"{bdir}/samples/samples_step_*.pt"))
    if not samples_files:
        return None, None
    steps, var_list = [], []
    for p in samples_files:
        step = int(os.path.basename(p).replace("samples_step_","").replace(".pt",""))
        x = torch.load(p, weights_only=True).float()
        cov = torch.cov(x.T)
        var_k = (eigvec_t.float().T @ cov @ eigvec_t.float()).diag().numpy()[:n_modes]
        steps.append(step)
        var_list.append(var_k)
    return np.array(steps), np.stack(var_list)

# Load eigenbasis from one pilot dir
eb_path = f"{list(beta_dirs.values())[0]}/eigenbasis.pt"
if os.path.exists(eb_path):
    eb = torch.load(eb_path, weights_only=True)
    eigval_pilot = eb["eigval"].float().numpy()
    eigvec_pilot = eb["eigvec"].float()
    print(f"Eigenbasis loaded: d={len(eigval_pilot)}, λ_max={eigval_pilot[0]:.3f}")

    colors_pilot = {"-2.0":"#d62728","-1.0":"#ff7f0e","0.0":"#2ca02c","+1.0":"#1f77b4"}
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    for ax, (beta_str, bdir) in zip(axes, beta_dirs.items()):
        steps_p, var_traj_p = load_var_traj(bdir, eigvec_pilot, n_modes=200)
        if steps_p is None:
            ax.text(0.5, 0.5, 'No data yet', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_title(f'β={beta_str}')
            continue

        ratio = var_traj_p / (eigval_pilot[:200][None,:] + 1e-10)
        im = ax.imshow(ratio.T, aspect='auto', origin='upper',
                       norm=LogNorm(vmin=0.1, vmax=5.0), cmap='RdYlGn',
                       extent=[steps_p[0], steps_p[-1], 200, 0])
        ax.set(xlabel='Step', ylabel='Mode k', title=f'β={beta_str}')
        plt.colorbar(im, ax=ax, label='var_k/λ_k')

    plt.suptitle('DINOv2 norm/patch pilot (d=768) — covariance heatmap (in progress)', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Pilot data not available — run the training first.")

---
## 9. File Structure Summary

```
Loss-Weighting-Spectral-Bias/
├── step1_validation/
│   ├── config.py               # constants, sweep grid, sigma grid helpers
│   ├── models.py               # LinearDenoiserShared, MLPDenoiser
│   ├── losses.py               # DeterministicPowerLawLoss, GeneralWeightingLoss
│   ├── eval.py                 # sampling-based covariance eval, emergence detection
│   ├── dinov2_gaussian_dataset.py  # load DINOv2 cov, sample synthetic Gaussian
│   ├── run_simple_test.py      # d=50 sanity check across β schedules
│   ├── run_mlp_dinov2.py       # full DINOv2 Gaussian training with log-scale callbacks
│   ├── analyze_mlp_dinov2.py   # posthoc: load samples → covariance traj → emergence
│   ├── run_per_sigma.py        # per-sigma analytic Phi integral computation
│   ├── per_sigma_analytic_d200_v2.json  # analytic α_phi results
│   └── DIAGNOSIS_beta2p5_nan.md         # why β=2.5 gives NaN
├── scripts/
│   ├── run_mlp_dinov2.sh       # Slurm array job for pilot training
│   └── simple_test_save.sh     # Slurm job for d=50 sanity check
├── figures/dinov2_covariance/  # DINOv2 eigenspectrum figures
└── ANALYSIS_dinov2_covariance.md  # full DINOv2 analysis report
```

**Store directory** (`$STORE/step1_results/`):
```
simple_test_d50_alpha1/    ← d=50 sanity check results + plots
mlp_dinov2/                ← pilot training (4 β values, 50k steps each)
mlp_dinov2_plots/          ← pilot analysis plots
slurm_logs/                ← job stdout/stderr
```

---
## 10. Next Steps

1. **Wait for pilot to finish** (~50k steps, ~2h total). Check convergence for β ∈ {-1.0, 0.0, +1.0} (skip β=-2.0 based on d=50 findings — it diverges).

2. **Run emergence analysis** (`analyze_mlp_dinov2.py`): fit power-law to τ_k* vs λ_k, extract α_trained.

3. **Compare theory vs training:** α_phi (from `per_sigma_analytic_d200_v2.json`) vs α_trained. The DINOv2 data has α_data≈0.56, which should give a specific α_phi prediction.

4. **EDM preconditioning:** Add `EDMPrecondWrapper` to the MLP and re-run — this may fix the β=-2.0 divergence by properly normalizing the denoiser output across σ levels.

5. **Sweep over α_data:** Test synthetic Gaussians with α_data ∈ {0.56, 1.0, 2.0} for each β to map the full theory-experiment comparison grid.